# 01 · The ML Workflow, end to end

This notebook mirrors the **ML Workflow** page of the demo app. You will go from a
messy CSV all the way to a saved, reusable model.

**You will learn to:**

1. load and inspect a dataset,
2. clean it (duplicates, missing values, messy text),
3. choose a target and build features (including one-hot encoding),
4. explore it with simple plots,
5. train and evaluate a classifier, and
6. wrap everything in a `Pipeline`, save it, and predict on new data.

Run each cell with **Shift+Enter**. Need a dataset? `data/customers.csv` ships with the repo.

## 1. Set up the tools

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

DATA = Path("..") / "data"
MODELS = Path("..") / "models"
print("pandas", pd.__version__)

## 2. Load the data

The question we want to answer: **will this customer churn (cancel)?**

In [ ]:
df = pd.read_csv(DATA / "customers.csv")
print("shape:", df.shape)
df.head()

### Inspect before you trust it

Look at column types, missing values and duplicate rows.

In [ ]:
df.info()
print("\nmissing values:\n", df.isna().sum())
print("\nduplicate rows:", df.duplicated().sum())

## 3. Clean the data

Real data is messy. We will:

- remove duplicate rows,
- trim + lowercase text (so `"Month-to-month"` == `"month-to-month"`),
- drop rows with missing values, and
- drop any constant columns (they carry no signal).

In [ ]:
clean = df.copy()
clean = clean.drop_duplicates()

for col in clean.columns:
    if clean[col].dtype.kind in "OUS":            # text-like column
        clean[col] = clean[col].str.strip().str.lower()

clean = clean.dropna()

# drop constant columns
clean = clean.loc[:, clean.nunique() > 1]

print("rows after cleaning:", len(clean))
print("missing left:", int(clean.isna().sum().sum()))

## 4. Choose a target and build features

`churn` is what we predict. Everything else (except the `customer_id`) is an input.
Text columns get **one-hot encoded** into 0/1 columns.

In [ ]:
target = "churn"
feature_cols = [c for c in clean.columns if c not in (target, "customer_id")]

X = clean[feature_cols].copy()
text_cols = [c for c in X.columns if X[c].dtype.kind in "OUS"]
X = pd.get_dummies(X, columns=text_cols)

y = (clean[target] == "yes").astype(int)          # 1 = churned, 0 = stayed

print("X:", X.shape, "| churn rate:", round(y.mean(), 3))
X.head()

## 5. Explore (EDA)

Always look at the data before modeling.

In [ ]:
y.value_counts().sort_index().plot(kind="bar", title="Class balance (1 = churned)")
plt.xlabel("churn")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(X.select_dtypes("number").corr(), cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation between features")
plt.show()

## 6. Split, train, evaluate

We score on a **test set** the model never saw — the only honest estimate.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print("accuracy:", round(accuracy_score(y_test, pred), 3))

In [ ]:
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots()
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_xlabel("predicted")
ax.set_ylabel("actual")
plt.show()

In [ ]:
print(classification_report(y_test, pred, zero_division=0))

## 7. Try other models

Same data, different algorithms. Which wins?

In [ ]:
models = {
    "Logistic regression": LogisticRegression(max_iter=1000),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Decision tree": DecisionTreeClassifier(random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    candidate = Pipeline([("scaler", StandardScaler()), ("clf", model)])
    candidate.fit(X_train, y_train)
    acc = accuracy_score(y_test, candidate.predict(X_test))
    print(f"{name:22s} accuracy = {acc:.3f}")

## 8. Save, reload, and predict

A `Pipeline` bundles preprocessing **and** the model, so new data is treated the same way.

In [ ]:
MODELS.mkdir(exist_ok=True)
joblib.dump(pipe, MODELS / "notebook_pipeline.joblib")

reloaded = joblib.load(MODELS / "notebook_pipeline.joblib")
print("reloaded accuracy:", round(accuracy_score(y_test, reloaded.predict(X_test)), 3))

In [ ]:
# Predict a brand-new customer (columns must match X)
new_customer = pd.DataFrame([{col: 0 for col in X.columns}])
new_customer["tenure_months"] = 3
new_customer["monthly_charges"] = 95.0
if "contract_month-to-month" in X.columns:
    new_customer["contract_month-to-month"] = 1

prob = reloaded.predict_proba(new_customer)[0, 1]
print(f"churn probability: {prob:.0%}")

## Your turn 🏋️

1. Try **filling** missing values with the median instead of dropping rows. Does accuracy change?
2. Add a new feature, e.g. `total_charges / tenure_months`.
3. Change `test_size` to `0.4` and see how the score moves.
4. Remove `support_calls` from the features — is the model worse?
5. (Challenge) Which feature seems most important? Try `model.feature_importances_` with the Random forest.